In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
import pandas as pd
import spacy
import re
import os
from tqdm import tqdm

# 1. Gestión de Rutas (Paths)
# Como el notebook está en /notebooks, subimos un nivel y entramos en /datasets
BASE_DIR = os.path.dirname(os.getcwd()) # Sube a la raíz del proyecto
DATASET_IN = os.path.join(BASE_DIR, "datasets", "dataset_models.csv")
DATASET_OUT = os.path.join(BASE_DIR, "datasets", "dataset_preprocesado.csv")

# 2. Funciones de limpieza con detección de Formato (Markdown/Estructura)
def limpiar_texto_estructural(texto):
    if not isinstance(texto, str): return ""
    
    # Marcamos negritas, listas y saltos de línea con tokens identificables
    texto = re.sub(r'\*\*(.*?)\*\*', r' BOLDTAG \1 BOLDTAG ', texto)
    texto = re.sub(r'^\s*\d+[\.\)]\s+', ' NUMLISTTAG ', texto, flags=re.MULTILINE)
    texto = re.sub(r'^\s*[\*\-\•]\s+', ' BULLETTAG ', texto, flags=re.MULTILINE)
    texto = texto.replace('\n', ' NEWLINETAG ')
    
    # Limpieza: quitamos URLs y carácteres extraños, pero protegemos nuestros TAGS
    texto = re.sub(r'http\S+|www\S+', '', texto)
    # Dejamos letras, puntuación y espacio (el TAG se mantendrá porque son letras)
    texto = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚñÑüÜ.,; ]', '', texto)
    
    return " ".join(texto.split())

def ejecutar_preprocesado():
    # Cargar spaCy (solo lo que necesitamos)
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "attribute_ruler"])
    
    if not os.path.exists(DATASET_IN):
        print(f"Error: No se encuentra el archivo en {DATASET_IN}")
        return

    print("Cargando dataset desde /datasets...")
    df = pd.read_csv(DATASET_IN)
    
    print("Detectando estructuras (negritas, listas, saltos)...")
    df['text_clean'] = df['text'].apply(limpiar_texto_estructural)
    
    print("Lematizando con nlp.pipe (Modo Turbo)...")
    textos_finales = []
    
    for doc in tqdm(nlp.pipe(df['text_clean'].astype(str), n_process=8, batch_size=1000), total=len(df)):
        # Lematizamos todo (incluyendo los TAGS que hemos creado)
        tokens = [token.lemma_.lower() for token in doc]
        textos_finales.append(" ".join(tokens))
    
    df['text_lemmatized'] = textos_finales
    
    # Guardar en la carpeta de datasets
    df.to_csv(DATASET_OUT, index=False)
    print(f"Archivo guardado en: {DATASET_OUT}")

if __name__ == "__main__":
    ejecutar_preprocesado()

Cargando dataset desde /datasets...
Detectando estructuras (negritas, listas, saltos)...
Lematizando con nlp.pipe (Modo Turbo)...


100%|██████████| 45481/45481 [05:34<00:00, 135.86it/s]


Archivo guardado en: c:\Users\aleja\Documents\Comillas\2º Cuatri\Análisis de Datos no Estructurados\TextClassification\datasets\dataset_preprocesado_estructural.csv
